# F26 DIIG NC Hotel Group Data Challenge
## Booking-time cancellation/no-show exposure
**Presenter:** Joshua Wu · **NetID:** jw1059

### Central question
At booking time, which reservations create the largest avoidable cancellation/no-show exposure, after bracketing duplicate records, and which reversible confirmation/deposit pilot should NCHG test?

**Guardrail:** ADR × stay nights is a gross exposure proxy—not lost revenue. This observational file cannot prove causal policy effects or safe overbooking.

In [1]:
from pathlib import Path
import hashlib, pandas as pd, numpy as np
INPUT=Path('data/nc_hotel_group.csv')
SEED=42
print('SHA256:', hashlib.sha256(INPUT.read_bytes()).hexdigest())
df=pd.read_csv(INPUT); print('shape:',df.shape); display(df.head(3))

SHA256: e8ea09acb1b3d45dfd7e5392373c433dc715b44939a0993ef54410f94d9e799d
shape: (20000, 24)


,property,canceled,advance_time,arrival_year,arrival_month,arrival_week_number,arrival_day_of_month,weekend_stays,week_stays,num_adults,...,num_cancellations,num_not_canceled,reserved_room_code,assigned_room_code,num_changes,deposit_terms,wait_days,average_daily_rate,status,status_date
0,Resort Hotel,0,203,2016,December,49,2,2,5,2,...,0,0,F,F,4,No Deposit,0,66.8,Check-Out,2016-12-09
1,City Hotel,1,82,2015,July,29,16,0,3,2,...,0,0,A,A,0,No Deposit,0,76.5,Canceled,2015-07-16
2,City Hotel,0,25,2016,December,53,27,0,3,3,...,0,0,A,K,2,No Deposit,0,60.0,Check-Out,2016-12-30


## Audit and definitions
The row is a reservation record, not a room-night. We preserve exact duplicates and report de-duplicated sensitivity. The outcome combines `Canceled` and `No-Show` because the supplied flag exactly matches status; no-show is separately reported.

In [2]:
df['arrival_date']=pd.to_datetime(df.arrival_year.astype(str)+'-'+df.arrival_month.astype(str)+'-'+df.arrival_day_of_month.astype(str))
df['stay_nights']=df.weekend_stays+df.week_stays
df['gross_proxy']=df.average_daily_rate*df.stay_nights
df['label']=df.status.isin(['Canceled','No-Show'])
print('missing cells:',int(df.isna().sum().sum()),'duplicates beyond first:',int(df.duplicated().sum()))
print(df.groupby('status').size()); print('label rate:',df.label.mean())

missing cells: 0 duplicates beyond first: 3486
status
Canceled      7348
Check-Out    12462
No-Show        190
dtype: int64
label rate: 0.3769


## Descriptive findings
City Hotel is higher-risk than Resort Hotel. Risk rises sharply with lead time. Groups and long-lead bookings form concentrated high-risk interactions. Non Refund is an unusually strong observational marker and must be validated before any policy change.

In [3]:
display(df.groupby('property').agg(bookings=('label','size'),rate=('label','mean'),proxy=('gross_proxy','sum')).sort_values('rate',ascending=False))
df['lead_band']=pd.cut(df.advance_time,[-1,0,7,30,90,180,9999],labels=['0','1–7','8–30','31–90','91–180','181+'])
display(df.groupby('lead_band',observed=True).agg(bookings=('label','size'),rate=('label','mean'),proxy=('gross_proxy','sum')))

,bookings,rate,proxy
property,,,
City Hotel,13203,0.427251,4180051.74
Resort Hotel,6797,0.279094,2980778.70


,bookings,rate,proxy
lead_band,,,
0,1045,0.066986,134247.79
1–7,2222,0.107561,490508.66
8–30,3154,0.279962,1063853.04
31–90,4945,0.378564,1908245.95
91–180,4473,0.456517,2006580.87
181+,4161,0.584475,1557394.13


## Time-aware model and sensitivity
The supplied durable analysis bundle fits regularized logistic regression with a chronological 70/15/15 split, excluding post-outcome fields. Final test AUC is 0.733 and Brier score 0.197 versus 0.239 prevalence baseline; performance drifts, so calibration monitoring is required. Exact de-duplication changes the headline rate from 37.69% to 31.00% and proxy exposure from $2.815m to $2.219m.

In [4]:
bundle=Path('data/evidence')
for f in ['model_metrics.csv','sensitivity.csv','duplicate_impact.csv','transparent_rule_eval.csv']:
 print('\n',f); display(pd.read_csv(bundle/f).head(12))


 model_metrics.csv


,split,n,start,end,cancel_rate,roc_auc,average_precision,brier,log_loss,mean_predicted,baseline_brier,baseline_log_loss
0,train,14000,2015-07-01,2017-02-10,0.367500,0.829661,0.803395,0.142837,0.445445,0.367558,0.232444,0.657612
1,validation,3000,2017-02-10,2017-05-22,0.400333,0.806029,0.789846,0.165291,0.489064,0.409824,0.240067,0.673147
2,test,3000,2017-05-22,2017-08-31,0.397333,0.732950,0.688201,0.197260,0.569804,0.441367,0.239460,0.671916



 sensitivity.csv


,sample,n,cancel_rate,no_show_rate,cancel_or_noshow_rate,stay_nights,gross_booked_proxy,gross_cancel_or_noshow_proxy
0,all,20000,0.376900,0.009500,0.376900,68740,7160830.44,2814874.51
1,exclude_zero_guest,19967,0.377323,0.009516,0.377323,68636,7159520.90,2814874.51
2,exclude_nonpositive_adr,19707,0.380931,0.009641,0.380931,68251,7160830.44,2814874.51
3,exclude_both,19704,0.380989,0.009643,0.380989,68236,7159520.90,2814874.51
4,adr_capped_p99,20000,0.376900,0.009500,0.376900,68740,7130244.93,2802175.82
5,exact_deduplicated,16514,0.309979,0.010537,0.309979,59126,6299135.84,2218521.19



 duplicate_impact.csv


,population,n,cancel_rate,no_show_rate,stay_nights,gross_cancel_proxy
0,all rows,20000,0.376900,0.009500,68740,2814874.51
1,singleton rows,15176,0.286769,0.011004,55231,2029521.36
2,rows in duplicate groups,4824,0.660448,0.004768,13509,785353.15



 transparent_rule_eval.csv


,split,n,base_cancel_rate,rule_n,rule_share,rule_cancel_rate,rule_lift,rule_capture_of_cancels
0,test,3000,0.397333,222,0.074000,1.000000,2.516779,0.186242
1,train,14000,0.367500,1935,0.138214,0.993282,2.702807,0.373567
2,valid,3000,0.400333,397,0.132333,1.000000,2.497918,0.330558


## Action
Use the model and transparent rule for monitoring and pilot targeting—not automatic rejection. Pre-register a randomized confirmation/deposit offer test among eligible long-lead/high-risk bookings, blocked by property, channel, lead band, and arrival month. Measure conversion, cancellations, kept stay nights, refunds/chargebacks, service contacts, and net contribution. Collect inventory, room counts, commissions, replacement demand, and walk costs before overbooking or channel-allocation decisions.